In [1]:
# imports
import time
import pandas as pd
import requests
from datetime import datetime
from riotwatcher import LolWatcher, ApiError
from pathlib import Path

# Walk up from cwd until we find the repo root (identified by .git folder).
# This works regardless of where Jupyter was launched from.
def _find_repo_root():
    p = Path().resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / ".git").exists():
            return candidate
    return p  # fallback: cwd

ROOT = _find_repo_root()
print(f"ROOT: {ROOT}")


ROOT: C:\Users\Laser\cs163-group18-winfactors


In [ ]:
import time
import requests
import pandas as pd
from collections import deque
from pathlib import Path
from riotwatcher import LolWatcher, ApiError

def _find_repo_root():
    p = Path().resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / ".git").exists():
            return candidate
    return p

ROOT    = _find_repo_root()
OUT_DIR = ROOT / "data" / "match_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUT_DIR: {OUT_DIR}")

# ── Region config ──────────────────────────────────────────────────────────────
# platform → routing region for match/matchlist endpoints
PLATFORM_TO_REGION = {
    "na1":  "americas",
    "euw1": "europe",
    "kr":   "asia",
}

def match_routing(mid):
    """Infer routing region from match ID prefix."""
    if mid.startswith("NA1_"):  return "americas"
    if mid.startswith("EUW1_"): return "europe"
    if mid.startswith("KR_"):   return "asia"
    if mid.startswith("EUN1_"): return "europe"
    return "americas"

# ── API key ────────────────────────────────────────────────────────────────────
def load_key():
    for p in [
        Path.cwd() / "riot_api_key.txt",
        Path.cwd() / "src" / "riot_api_key.txt",
        ROOT / "riot_api_key.txt",
        ROOT / "src" / "riot_api_key.txt",
    ]:
        if p.exists():
            return p.read_text().strip()
    raise FileNotFoundError("riot_api_key.txt not found.")

api_key = load_key()
watcher = LolWatcher(api_key)
print(f"Loaded API key: {api_key[:10]}... (length: {len(api_key)})")

# ── Exceptions ─────────────────────────────────────────────────────────────────
class KeyExpiredError(Exception):
    pass

class RateLimitError(Exception):
    def __init__(self, wait):
        self.wait = wait  # seconds to wait before retrying this region

# ── Parsing ────────────────────────────────────────────────────────────────────
def parse_match(match_json):
    meta = match_json.get("metadata", {})
    info = match_json.get("info", {})
    match_id = meta.get("matchId") or match_json.get("gameId")
    team_map = {}
    for t in info.get("teams", []):
        tid = t.get("teamId")
        obj = t.get("objectives", {})
        team_map[tid] = {
            "team_win":               t.get("win", False),
            "team_baron_kills":       obj.get("baron",      {}).get("kills", 0),
            "team_dragon_kills":      obj.get("dragon",     {}).get("kills", 0),
            "team_tower_kills":       obj.get("tower",      {}).get("kills", 0),
            "team_inhibitor_kills":   obj.get("inhibitor",  {}).get("kills", 0),
            "team_rift_herald_kills": obj.get("riftHerald", {}).get("kills", 0),
        }
    rows = []
    for p in info.get("participants", []):
        tid = p.get("teamId")
        team = team_map.get(tid, {})
        kills, deaths, assists = p.get("kills", 0), p.get("deaths", 0), p.get("assists", 0)
        rows.append({
            "match_id":                  match_id,
            "puuid":                     p.get("puuid"),
            "summonerName":              p.get("riotIdGameName") or p.get("summonerName"),
            "championName":              p.get("championName"),
            "championId":                p.get("championId"),
            "teamId":                    tid,
            "teamPosition":              p.get("teamPosition"),
            "role":                      p.get("role"),
            "win":                       p.get("win"),
            "kills":                     kills,
            "deaths":                    deaths,
            "assists":                   assists,
            "kda":                       (kills + assists) / max(1, deaths),
            "totalDamageToChampions":    p.get("totalDamageDealtToChampions", 0),
            "physicalDamageToChampions": p.get("physicalDamageDealtToChampions", 0),
            "magicDamageToChampions":    p.get("magicDamageDealtToChampions", 0),
            "totalDamageTaken":          p.get("totalDamageTaken", 0),
            "damageSelfMitigated":       p.get("damageSelfMitigated", 0),
            "damageToBuildings":         p.get("damageDealtToBuildings", 0),
            "damageToObjectives":        p.get("damageDealtToObjectives", 0),
            "goldEarned":                p.get("goldEarned", 0),
            "goldSpent":                 p.get("goldSpent", 0),
            "totalMinionsKilled":        p.get("totalMinionsKilled", 0),
            "neutralMinionsKilled":      p.get("neutralMinionsKilled", 0),
            "cs":                        p.get("totalMinionsKilled", 0) + p.get("neutralMinionsKilled", 0),
            "visionScore":               p.get("visionScore", 0),
            "wardsPlaced":               p.get("wardsPlaced", 0),
            "wardsKilled":               p.get("wardsKilled", 0),
            "visionWardsBought":         p.get("visionWardsBoughtInGame", 0),
            "timeCCingOthers":           p.get("timeCCingOthers", 0),
            "totalTimeCCDealt":          p.get("totalTimeCCDealt", 0),
            "champLevel":                p.get("champLevel"),
            "doubleKills":               p.get("doubleKills", 0),
            "tripleKills":               p.get("tripleKills", 0),
            "quadraKills":               p.get("quadraKills", 0),
            "pentaKills":                p.get("pentaKills", 0),
            "turretKills":               p.get("turretKills", 0),
            "firstBloodKill":            p.get("firstBloodKill", False),
            "firstBloodAssist":          p.get("firstBloodAssist", False),
            "longestTimeSpentLiving":    p.get("longestTimeSpentLiving"),
            "totalTimeSpentDead":        p.get("totalTimeSpentDead"),
            "spell1Casts":               p.get("spell1Casts", 0),
            "spell2Casts":               p.get("spell2Casts", 0),
            "spell3Casts":               p.get("spell3Casts", 0),
            "spell4Casts":               p.get("spell4Casts", 0),
            "team_win":                  team.get("team_win"),
            "team_baron_kills":          team.get("team_baron_kills", 0),
            "team_dragon_kills":         team.get("team_dragon_kills", 0),
            "team_tower_kills":          team.get("team_tower_kills", 0),
            "team_inhibitor_kills":      team.get("team_inhibitor_kills", 0),
            "team_rift_herald_kills":    team.get("team_rift_herald_kills", 0),
        })
    return rows

# ── Fetch helpers (raise RateLimitError instead of sleeping) ───────────────────
def fetch_match(region, mid):
    try:
        return watcher.match.by_id(region, mid)
    except ApiError as e:
        code = e.response.status_code
        if code == 401: raise KeyExpiredError()
        if code == 429:
            wait = int(e.response.headers.get("Retry-After", 10))
            raise RateLimitError(wait)
        print(f"  API error {code} for {mid}, skipping.")
        return None
    except KeyExpiredError:
        raise
    except Exception as exc:
        print(f"  Error fetching {mid}: {exc}")
        return None

def get_match_ids(puuid, region, max_matches=None):
    ids, start = [], 0
    while True:
        try:
            batch = watcher.match.matchlist_by_puuid(region, puuid, start=start, count=100)
        except ApiError as e:
            if e.response.status_code == 401: raise KeyExpiredError()
            if e.response.status_code == 429:
                wait = int(e.response.headers.get("Retry-After", 10))
                raise RateLimitError(wait)
            break
        if not batch:
            break
        ids.extend(batch)
        if max_matches and len(ids) >= max_matches:
            ids = ids[:max_matches]
            break
        start += 100
        time.sleep(0.4)
    return ids

def fetch_tier_entries(platform, tier, api_key):
    endpoints = {
        "challenger":  f"https://{platform}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5",
        "grandmaster": f"https://{platform}.api.riotgames.com/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5",
        "master":      f"https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5",
    }
    r = requests.get(endpoints[tier.lower()], headers={"X-Riot-Token": api_key})
    r.raise_for_status()
    entries = r.json().get("entries", [])
    print(f"    {tier.capitalize()}: {len(entries)} players")
    return entries

# ── Multi-region collection ────────────────────────────────────────────────────
# Maintains a separate player queue per region. When a region hits a rate limit,
# switches to another region's queue instead of sleeping. Only sleeps when ALL
# regions are cooling down simultaneously.
def collect(
    platforms=(("na1", "americas"), ("euw1", "europe"), ("kr", "asia")),
    tiers=("challenger", "grandmaster", "master"),
    max_players_per_region=None,
    max_matches_per_player=None,
    out_file="challenger_matches.csv",
    save_every=1000,
    print_every=10,
):
    out_path = OUT_DIR / out_file

    if out_path.exists():
        existing = pd.read_csv(out_path)
        all_rows = existing.to_dict("records")
        seen = set(existing["match_id"].dropna().unique())
        print(f"Resuming: {len(seen)} matches already collected ({len(all_rows)} rows)")
    else:
        all_rows, seen = [], set()
        print(f"Starting fresh → {out_path}")

    # ── Build per-region player queues ─────────────────────────────────────────
    queues = {}          # region → deque of {puuid, platform, tier}
    rl_until = {}        # region → timestamp when rate limit expires

    for platform, region in platforms:
        print(f"\n{platform.upper()} ({region}):")
        entries, seen_puuids = [], set()
        for tier in tiers:
            try:
                for e in fetch_tier_entries(platform, tier, api_key):
                    uid = e.get("puuid") or e.get("summonerId") or id(e)
                    if uid not in seen_puuids:
                        entries.append({"puuid": e.get("puuid"),
                                        "summonerId": e.get("summonerId"),
                                        "platform": platform,
                                        "region": region,
                                        "_tier": tier})
                        seen_puuids.add(uid)
            except Exception as exc:
                print(f"    Failed to fetch {tier}: {exc}")

        if max_players_per_region:
            entries = entries[:max_players_per_region]
        queues[region]   = deque(entries)
        rl_until[region] = 0
        print(f"  → {len(entries)} unique players queued")

    total_players = sum(len(q) for q in queues.values())
    print(f"\nTotal players across all regions: {total_players}")

    # ── Main loop ──────────────────────────────────────────────────────────────
    all_done = False
    processed = 0

    def save():
        pd.DataFrame(all_rows).to_csv(out_path, index=False)

    def pick_region():
        """Return the next available region, or None if all are cooling down."""
        now = time.time()
        available = [(rl_until[r], r) for r in queues if queues[r] and rl_until[r] <= now]
        if not available:
            return None
        return min(available)[1]  # region with earliest rate-limit expiry

    while any(queues.values()):
        region = pick_region()

        if region is None:
            # All regions cooling — sleep until the soonest one opens
            active_regions = [r for r in queues if queues[r]]
            wake = min(rl_until[r] for r in active_regions)
            sleep_for = max(0.1, wake - time.time())
            print(f"\n  All regions rate-limited — sleeping {sleep_for:.1f}s...")
            time.sleep(sleep_for)
            continue

        entry    = queues[region].popleft()
        platform = entry["platform"]
        puuid    = entry.get("puuid")
        processed += 1

        if not puuid:
            try:
                summ = watcher.summoner.by_id(platform, entry["summonerId"])
                puuid = summ["puuid"]
                time.sleep(0.4)
            except KeyExpiredError:
                save()
                print(f"\nKey expired. Saved {len(seen)} matches. Update riot_api_key.txt and re-run.")
                return None
            except Exception as exc:
                print(f"Couldn't resolve puuid: {exc}")
                continue

        print(f"\n[{entry['_tier']}@{platform}] player {processed}/{total_players}: {puuid[:12]}...")

        # Fetch match IDs — handle rate limit by pushing player back and switching
        try:
            match_ids = get_match_ids(puuid, region, max_matches=max_matches_per_player)
        except RateLimitError as e:
            print(f"  Rate limit on {region} ({e.wait}s) — switching region")
            rl_until[region] = time.time() + e.wait
            entry["puuid"] = puuid
            queues[region].appendleft(entry)  # put back at front
            continue
        except KeyExpiredError:
            save()
            print(f"\nKey expired. Saved {len(seen)} matches. Update riot_api_key.txt and re-run.")
            return None

        new_ids = [m for m in match_ids if m not in seen]
        print(f"  {len(match_ids)} match IDs, {len(new_ids)} new")
        if not new_ids:
            continue

        for j, mid in enumerate(new_ids, 1):
            # Match routing is inferred from the match ID prefix
            mid_region = match_routing(mid)
            while True:
                try:
                    data = fetch_match(mid_region, mid)
                    break
                except RateLimitError as e:
                    print(f"  Rate limit on {mid_region} ({e.wait}s) — waiting (match fetch)")
                    # Can't switch region for match fetches — the match lives on one region
                    time.sleep(e.wait + 1)
                except KeyExpiredError:
                    save()
                    print(f"\nKey expired. Saved {len(seen)} matches. Update riot_api_key.txt and re-run.")
                    return None

            if not data:
                continue
            try:
                rows = parse_match(data)
                all_rows.extend(rows)
                seen.add(mid)
            except Exception as exc:
                print(f"  Parse error {mid}: {exc}")
            time.sleep(0.4)

            total = len(seen)
            if total % print_every == 0:
                print(f"  [{total} matches] {platform} player {processed}, match {j}/{len(new_ids)}: {mid}")
            if total % save_every == 0:
                save()
                print(f"  ── saved {total} matches ──")

        time.sleep(0.5)

    save()
    df = pd.DataFrame(all_rows)
    print(f"\nDone. {len(seen)} matches, {len(df)} participant rows → {out_path}")
    return df

# ── Run ────────────────────────────────────────────────────────────────────────
df = collect(
    platforms=(("na1", "americas"), ("euw1", "europe"), ("kr", "asia")),
    tiers=("challenger", "grandmaster", "master"),
    max_matches_per_player=30,   # cap per player to keep runtime reasonable
    out_file="challenger_matches.csv",
)


In [ ]:
# Test the API key with a simple request
# Run this cell to verify your Riot API key works before running the main pipeline.

import requests

try:
    api_key = load_key()
    print(f"Testing key: {api_key[:10]}... (length: {len(api_key)})")
    headers = {"X-Riot-Token": api_key}
    # Test with a known summoner (change 'Doublelift' to any valid name if needed)
    test_url = "https://na1.api.riotgames.com/lol/summoner/v4/summoners/by-name/Doublelift"
    r = requests.get(test_url, headers=headers)
    print(f"Response status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Success! Summoner: {data.get('name')}, Level: {data.get('summonerLevel')}")
    else:
        print(f"Failed: {r.text}")
except Exception as e:
    print(f"Error testing key: {e}")